# Train Tokenizer **indmixedtufs4** 10.000 Vocab

# Import Library

In [1]:
from tokenizers import ByteLevelBPETokenizer

# Initialize a Byte-Pair Encoding(BPE) tokenizer
tokenizer = ByteLevelBPETokenizer()

print(tokenizer)

Tokenizer(vocabulary_size=0, model=ByteLevelBPE, add_prefix_space=False, lowercase=False, dropout=None, unicode_normalizer=None, continuing_subword_prefix=None, end_of_word_suffix=None, trim_offsets=False)


# Load Corpus

In [2]:
training_dataset = ('/kaggle/input/korpus-indomixedtufs4-165k-sentence/clean-indmixtufs4-165k-sent.txt')

print(training_dataset)

/kaggle/input/korpus-indomixedtufs4-165k-sentence/clean-indmixtufs4-165k-sent.txt


# Train Tokenizer

In [3]:
# Latih tokenizer

tokenizer.train(
    files=training_dataset, # data corpus
    vocab_size=10000,  # Ukuran total vocab asli BART 0-50264 total 50265 token
    min_frequency=2,   # Abaikan token yang muncul kurang dari 2 kali
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]  # Special token BART
)

In [4]:
print(f"Tokenizer setelah train: \n{tokenizer}")

Tokenizer setelah train: 
Tokenizer(vocabulary_size=10000, model=ByteLevelBPE, add_prefix_space=False, lowercase=False, dropout=None, unicode_normalizer=None, continuing_subword_prefix=None, end_of_word_suffix=None, trim_offsets=False)


In [5]:
import os

# Save the tokenizer
output_dir = "./indmixedtufs4-tokenizer"
os.makedirs(output_dir, exist_ok=True)
tokenizer.save_model(output_dir)

['./indmixedtufs4-tokenizer/vocab.json',
 './indmixedtufs4-tokenizer/merges.txt']

# Swap mask token index

Menyesuaikan index <mask> token, karena <mask> pada BART berda di index paing terakhir

In [6]:
import json

vocab_file = '/kaggle/working/indmixedtufs4-tokenizer/vocab.json'

# Baca vocab.json
with open(vocab_file, "r", encoding="utf-8") as f:
    vocab = json.load(f)

print("Ukuran vocabulary awal:", len(vocab))
print("Indeks <mask> awal:", vocab.get("<mask>"))

Ukuran vocabulary awal: 10000
Indeks <mask> awal: 4


In [7]:
# Check last index token
token_last_index = None
for token, idx in vocab.items():
    if idx == 9999:
        token_last_index = token
        break
print(f"Token di indeks terakhir: {token_last_index}")

Token di indeks terakhir: Ġama


In [8]:
# Tukar indeks: Pindahkan token di indeks terakhir ke 4
if token_last_index:
    vocab[token_last_index] = 4  # Pindahkan token lama ke indeks 4
else:
    print("Tidak ada token di indeks 9999")

# Tambahkan <mask> di indeks 29999
vocab["<mask>"] = 9999

In [16]:
# Check Index <mask>
print("Ukuran vocabulary setelah modifikasi:", len(vocab))
print("Indeks <mask> setelah modifikasi:", vocab.get("<mask>"))
print(f"Indeks {token_last_index} setelah modifikasi:", vocab.get(token_last_index))

Ukuran vocabulary setelah modifikasi: 10000
Indeks <mask> setelah modifikasi: 9999
Indeks Ġama setelah modifikasi: 4


In [10]:
# Periksa duplikasi <mask>
mask_count = sum(1 for token in vocab if token == "<mask>")
print(f"Jumlah <mask> di vocabulary: {mask_count}")

Jumlah <mask> di vocabulary: 1


In [11]:
# Sort vocab secara asc
sorted_vocab = dict(sorted(vocab.items(), key=lambda x: x[1]))

## Simpan vocab.json

In [12]:
# simpan vocab.json

# Direktori
new_output_dir = '/kaggle/working/indmixedtufs4-tokenizer-new'
os.makedirs(new_output_dir, exist_ok=True)

# Simpan output vocab.json
output_json_file = os.path.join(new_output_dir, "vocab.json")
with open(output_json_file, "w", encoding="utf-8") as f:
    json.dump(sorted_vocab, f, ensure_ascii=False, indent=2)
print(f"Output disimpan ke: {output_json_file}")

Output disimpan ke: /kaggle/working/indmixedtufs4-tokenizer-new/vocab.json


## Simpan merge.txt

In [13]:
import shutil

# Buat copy merge.txt dan simpan ke folder baru

merges_file = '/kaggle/working/indmixedtufs4-tokenizer/merges.txt'

merges_copy = '/kaggle/working/indmixedtufs4-tokenizer-new'

# Buat salinan merges.txt ke direktori baru
merges_copy_file = os.path.join(merges_copy, "merges.txt")

shutil.copy(merges_file, merges_copy_file)

print(f"Salinan merges.txt dibuat di: {merges_copy_file}")

Salinan merges.txt dibuat di: /kaggle/working/indmixedtufs4-tokenizer-new/merges.txt


In [14]:
print(os.listdir('/kaggle/working/indmixedtufs4-tokenizer-new'))

['merges.txt', 'vocab.json']


# Upload Tokenizer ke Kagglehub

In [17]:
import kagglehub

kagglehub.login()

# Tokenizer custom path
MY_TOKENIZER_DIR = '/kaggle/working/indmixedtufs4-tokenizer-new'

# Tokenizer name
TOKENIZER_SLUG = 'tokenizer-indomixedtufs4' # Nama tokenizer
VARIATION_SLUG = 'tokenizer-indomixedtufs4-10000' # Variasi tokenizer

# Upload model ke kagglehub
kagglehub.model_upload(
  handle = f"jawawahirul/{TOKENIZER_SLUG}/transformers/{VARIATION_SLUG}",
  local_model_dir = MY_TOKENIZER_DIR,
  version_notes = 'Update 2025-05-27'
)

Uploading Model https://www.kaggle.com/models/jawawahirul/tokenizer-indomixedtufs4/transformers/tokenizer-indomixedtufs4-10000 ...
Model 'tokenizer-indomixedtufs4' does not exist or access is forbidden for user 'jawawahirul'. Creating or handling Model...
Model 'tokenizer-indomixedtufs4' Created.
Starting upload for file /kaggle/working/indmixedtufs4-tokenizer-new/merges.txt


Uploading: 100%|██████████| 83.2k/83.2k [00:00<00:00, 193kB/s]

Upload successful: /kaggle/working/indmixedtufs4-tokenizer-new/merges.txt (81KB)
Starting upload for file /kaggle/working/indmixedtufs4-tokenizer-new/vocab.json



Uploading: 100%|██████████| 183k/183k [00:00<00:00, 308kB/s]

Upload successful: /kaggle/working/indmixedtufs4-tokenizer-new/vocab.json (179KB)


Your model instance has been created.
Files are being processed...
See at: https://www.kaggle.com/models/jawawahirul/tokenizer-indomixedtufs4/transformers/tokenizer-indomixedtufs4-10000
